# Graph Neural Networks: Visual Message Passing and a Real Comparison

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/02_deep_learning/notebooks/gnn_visual_message_passing_colab.ipynb)

This tutorial is designed as the GNN part of the deep learning lecture sequence:

1. neural networks and CNNs,
2. transformers,
3. graph neural networks.

The notebook has two goals:

- **Part A:** make one GNN update visible step by step. Students can see node features, edge weights, learnable weights, messages, aggregation, and updated node states.
- **Part B:** use a real graph dataset to compare a feature-only machine learning baseline with a GNN. The point is to make the benefit of neighbourhood information visible.

The code uses PyTorch and PyTorch Geometric so that the conceptual explanation is connected to standard GNN tooling.

## 0. Colab Setup

PyTorch is usually preinstalled in Colab. PyTorch Geometric is installed here because it provides standard graph datasets and layers.

In [ ]:
!pip -q install torch-geometric networkx matplotlib pandas scikit-learn ipywidgets pyvis

In [ ]:
import html as html_module
import math
import random
import tempfile
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import HTML, Math, clear_output, display
from pyvis.network import Network
from torch import nn
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.utils import k_hop_subgraph

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

# Part A: A GNN Update, Made Visible

A basic graph neural network layer updates each node by mixing information from its neighbours.

For a GCN-style layer, one update can be written as:

\[
H^{(1)} = \sigma\left(\hat{A} X W\right)
\]

where:

- \(X\) is the input node feature matrix,
- \(W\) is a learnable weight matrix,
- \(\hat{A}\) is the normalized adjacency matrix with self-loops,
- \(\sigma\) is a nonlinear activation function,
- \(H^{(1)}\) is the updated node representation.

The graph is tiny so every number can be inspected. The next interactive cell is the most important one: click **Update one layer** repeatedly and watch how values and colors change.

In [ ]:
display(Math(r"H^{(k+1)} = \mathrm{ReLU}\left(\hat{A} H^{(k)} W^{(k)}\right)"))
display(Math(r"\hat{A}_{ij} = \frac{1}{\sqrt{d_i d_j}}"))

In [ ]:
toy_edges = [
    (0, 1), (0, 2),
    (1, 2), (1, 3),
    (2, 4),
    (3, 4), (3, 5),
    (4, 5),
]

node_names = {
    0: "A: river",
    1: "B: river",
    2: "C: mixed",
    3: "D: street",
    4: "E: street",
    5: "F: street",
}

X0 = torch.tensor([
    [1.00, 0.10],
    [0.90, 0.20],
    [0.55, 0.55],
    [0.20, 0.90],
    [0.10, 1.00],
    [0.05, 0.95],
], dtype=torch.float32)

W0 = torch.tensor([
    [ 1.20, -0.70],
    [-0.35,  1.10],
], dtype=torch.float32)

G_toy = nx.Graph()
G_toy.add_edges_from(toy_edges)
pos_toy = nx.spring_layout(G_toy, seed=SEED)

In [ ]:
def normalized_adjacency(num_nodes, edges):
    A = torch.zeros((num_nodes, num_nodes), dtype=torch.float32)
    for i, j in edges:
        A[i, j] = 1
        A[j, i] = 1
    A = A + torch.eye(num_nodes)
    degree = A.sum(dim=1)
    d_inv_sqrt = torch.diag(torch.pow(degree, -0.5))
    A_hat = d_inv_sqrt @ A @ d_inv_sqrt
    return A, A_hat, degree


def feature_color(values):
    values = values.detach().cpu().numpy()
    return values[:, 0] - values[:, 1]


def draw_graph_state(features, title, edge_weights=None, cmap="coolwarm"):
    fig, ax = plt.subplots(figsize=(7, 5))
    scalar = feature_color(features)
    nodes = nx.draw_networkx_nodes(
        G_toy,
        pos_toy,
        node_color=scalar,
        cmap=cmap,
        vmin=-1.2,
        vmax=1.2,
        node_size=1150,
        edgecolors="#222",
        linewidths=1.2,
        ax=ax,
    )
    nx.draw_networkx_edges(G_toy, pos_toy, width=1.8, alpha=0.55, ax=ax)
    labels = {i: f"{i}\\n[{features[i,0]:.2f}, {features[i,1]:.2f}]" for i in G_toy.nodes}
    nx.draw_networkx_labels(G_toy, pos_toy, labels=labels, font_size=9, ax=ax)
    if edge_weights:
        nx.draw_networkx_edge_labels(G_toy, pos_toy, edge_labels=edge_weights, font_size=8, ax=ax)
    fig.colorbar(nodes, ax=ax, shrink=0.75, label="feature[0] - feature[1]")
    ax.set_title(title)
    ax.axis("off")
    plt.show()


def feature_table(features, title):
    df = pd.DataFrame(features.detach().cpu().numpy(), columns=["feature_0", "feature_1"])
    df.insert(0, "node", [node_names[i] for i in range(len(df))])
    print(title)
    display(df.round(3))


A_with_self_loops, A_hat, degree = normalized_adjacency(num_nodes=X0.size(0), edges=toy_edges)
feature_table(X0, "Initial node features X")
draw_graph_state(X0, "Initial node features")

## Interactive Message Passing App

This app is meant to be used before the static step-by-step cells below.

- Choose a target node.
- Red boundary = current target, yellow boundary = neighbouring message source, black boundary = not used in the selected update.
- The two small lights on each node show the two hidden dimensions; stronger color means a larger hidden value.
- Animated red dots move directly inside the graph along the edges used for message passing.
- Click **Update one layer** to apply one GCN-style update.
- The right-hand panel visualizes how source hidden states, the weight matrix, edge weights, message vectors, summation, and ReLU produce the next target state.

The graph state is \(H^{(k)}\). Node labels stay compact so students watch the hidden-state lights and message flow instead of reading numbers on every node.


In [ ]:
APP_WEIGHTS = [
    W0,
    torch.tensor([[0.80, 0.35], [0.25, 0.90]], dtype=torch.float32),
    torch.tensor([[0.65, -0.15], [0.20, 0.75]], dtype=torch.float32),
]

app_state = {
    "layer": 0,
    "features": X0.clone(),
    "last_update": None,
}

target_dropdown = widgets.Dropdown(
    options=[(f"{i}: {node_names[i]}", i) for i in range(X0.size(0))],
    value=3,
    description="Target",
)
update_button = widgets.Button(description="Update one layer", button_style="primary", icon="refresh")
reset_button = widgets.Button(description="Reset", button_style="", icon="undo")
app_output = widgets.Output()

NODE_POS = {
    0: (150, 115),
    1: (360, 95),
    2: (250, 265),
    3: (535, 250),
    4: (365, 425),
    5: (680, 410),
}

ROLE_STYLE = {
    "target": {"stroke": "#d62728", "width": 7, "label": "current target"},
    "neighbour": {"stroke": "#f0c419", "width": 6, "label": "neighbour / message source"},
    "inactive": {"stroke": "#111111", "width": 3, "label": "not used in selected update"},
}

LIGHT_COLORS = ["#22c55e", "#ef4444"]


def message_sources(target_node):
    return [int(src) for src in range(X0.size(0)) if A_hat[target_node, src] > 0]


def node_role(node, target_node, neighbours):
    if node == target_node:
        return "target"
    if node in neighbours:
        return "neighbour"
    return "inactive"


def clamp01(value):
    return max(0.0, min(1.0, float(value)))


def light_opacity(value):
    return 0.18 + 0.82 * clamp01(abs(value) / 1.25)


def light_radius(value):
    return 5 + 7 * clamp01(abs(value) / 1.25)


def format_value(value):
    return f"{float(value):.3f}"


def hidden_light_svg(values, x, y, scale=1.0):
    parts = []
    for idx, value in enumerate(values):
        cx = x + idx * 20 * scale
        radius = light_radius(value) * scale
        opacity = light_opacity(value)
        color = LIGHT_COLORS[idx]
        parts.append(
            f"<circle cx='{cx:.1f}' cy='{y:.1f}' r='{radius:.1f}' fill='{color}' fill-opacity='{opacity:.2f}' "
            f"stroke='#111827' stroke-width='{1.2 * scale:.1f}'><title>hidden_{idx}: {format_value(value)}</title></circle>"
        )
    return "".join(parts)


def scalar_chip(value, label=None, color="#22c55e"):
    opacity = light_opacity(value)
    text = format_value(value)
    label_html = f"<span class='chip-label'>{label}</span>" if label else ""
    return (
        f"<span class='scalar-chip'>"
        f"{label_html}<span class='chip-dot' style='background:{color}; opacity:{opacity:.2f}'></span>"
        f"<span class='chip-value'>{text}</span></span>"
    )


def vector_chip(values, title):
    dots = "".join(
        f"<span class='vector-dot' style='background:{LIGHT_COLORS[idx]}; opacity:{light_opacity(value):.2f}' "
        f"title='hidden_{idx}: {format_value(value)}'></span>"
        for idx, value in enumerate(values)
    )
    nums = " / ".join(format_value(v) for v in values)
    return f"<div class='vector-chip'><div class='mini-title'>{title}</div><div>{dots}</div><div class='mini-num'>{nums}</div></div>"


def message_table_for_target(features, weight, target_node):
    transformed = features @ weight
    rows = []
    for source_node in message_sources(target_node):
        coeff = A_hat[target_node, source_node].item()
        contribution = coeff * transformed[source_node]
        rows.append({
            "source_node": source_node,
            "source": f"{source_node}: {node_names[source_node]}",
            "edge_weight": coeff,
            "hidden": features[source_node].detach().cpu().numpy(),
            "after_w": transformed[source_node].detach().cpu().numpy(),
            "message": contribution.detach().cpu().numpy(),
        })
    return rows


def single_graph_html(features, target_node, layer):
    neighbours = set(message_sources(target_node))
    edge_parts = []
    active_edge_parts = []
    node_parts = []
    target_x, target_y = NODE_POS[target_node]

    for i, j in toy_edges:
        x1, y1 = NODE_POS[i]
        x2, y2 = NODE_POS[j]
        pair_active = target_node in (i, j) and (i in neighbours or j in neighbours)
        if pair_active:
            src = j if i == target_node else i
            sx, sy = NODE_POS[src]
            weight = A_hat[target_node, src].item()
            active_edge_parts.append(f"""
            <line x1='{sx}' y1='{sy}' x2='{target_x}' y2='{target_y}' stroke='#d62728' stroke-width='4.5' opacity='0.78' marker-end='url(#arrowhead)'/>
            <text x='{(sx + target_x) / 2:.1f}' y='{(sy + target_y) / 2 - 8:.1f}' class='edge-label'>{weight:.2f}</text>
            <circle r='7' fill='#d62728'>
              <animateMotion dur='1.35s' begin='{0.12 * len(active_edge_parts):.2f}s' repeatCount='indefinite' path='M {sx},{sy} L {target_x},{target_y}' />
            </circle>
            """)
        else:
            edge_parts.append(f"<line x1='{x1}' y1='{y1}' x2='{x2}' y2='{y2}' stroke='#cbd5e1' stroke-width='2.3'/>")

    self_weight = A_hat[target_node, target_node].item()
    active_edge_parts.append(f"""
    <circle cx='{target_x}' cy='{target_y}' r='49' fill='none' stroke='#d62728' stroke-width='3' opacity='0.0'>
      <animate attributeName='r' values='40;55;40' dur='1.35s' begin='0s' repeatCount='indefinite'/>
      <animate attributeName='opacity' values='0.15;0.85;0.15' dur='1.35s' begin='0s' repeatCount='indefinite'/>
    </circle>
    <text x='{target_x + 42}' y='{target_y - 40}' class='edge-label'>self {self_weight:.2f}</text>
    """)

    for node, (x, y) in NODE_POS.items():
        role = node_role(node, target_node, neighbours)
        style = ROLE_STYLE[role]
        node_values = features[node].detach().cpu().numpy()
        node_parts.append(f"""
        <g class='node'>
          <circle cx='{x}' cy='{y}' r='38' fill='#ffffff' stroke='{style['stroke']}' stroke-width='{style['width']}'/>
          <text x='{x}' y='{y + 6}' text-anchor='middle' class='node-id'>{node}</text>
          <text x='{x}' y='{y + 57}' text-anchor='middle' class='node-name'>{html_module.escape(node_names[node])}</text>
          <g transform='translate(22,-34)'>{hidden_light_svg(node_values, x, y, scale=0.88)}</g>
          <title>{node}: {node_names[node]} | {style['label']} | H^({layer}) = [{format_value(node_values[0])}, {format_value(node_values[1])}]</title>
        </g>
        """)

    return f"""
    <style>
      .graph-card {{ border: 1px solid #d1d5db; border-radius: 8px; background: #ffffff !important; color: #111827 !important; box-sizing: border-box; padding: 0; min-height: 590px; overflow: hidden; }}
      .node-id {{ font: 800 22px Arial, sans-serif; fill: #111827; }}
      .node-name {{ font: 600 12px Arial, sans-serif; fill: #334155; }}
      .edge-label {{ font: 800 13px Arial, sans-serif; fill: #b91c1c; paint-order: stroke; stroke: #ffffff; stroke-width: 4px; }}
      .legend text {{ font: 700 12px Arial, sans-serif; fill: #111827; }}
    </style>
    <div class='graph-card'>
      <svg viewBox='40 35 760 505' width='100%' height='590' role='img' aria-label='Animated message passing graph'>
        <defs>
          <marker id='arrowhead' markerWidth='11' markerHeight='9' refX='10' refY='4.5' orient='auto'>
            <path d='M0,0 L11,4.5 L0,9 z' fill='#d62728'></path>
          </marker>
          <filter id='softShadow' x='-20%' y='-20%' width='140%' height='140%'>
            <feDropShadow dx='0' dy='3' stdDeviation='3' flood-color='#94a3b8' flood-opacity='0.28'/>
          </filter>
        </defs>
        <rect x='48' y='42' width='744' height='488' rx='14' fill='#ffffff' stroke='#d1d5db'/>
        <g>{''.join(edge_parts)}</g>
        <g>{''.join(active_edge_parts)}</g>
        <g filter='url(#softShadow)'>{''.join(node_parts)}</g>
        <g transform='translate(65 470)' class='legend'>
          <text x='0' y='0'>boundary:</text>
          <circle cx='88' cy='-5' r='10' fill='#fff' stroke='#d62728' stroke-width='4'/><text x='104' y='0'>target</text>
          <circle cx='180' cy='-5' r='10' fill='#fff' stroke='#f0c419' stroke-width='4'/><text x='196' y='0'>neighbour</text>
          <circle cx='304' cy='-5' r='10' fill='#fff' stroke='#111111' stroke-width='3'/><text x='320' y='0'>other</text>
          <text x='430' y='0'>hidden state lights:</text>
          <circle cx='562' cy='-5' r='9' fill='{LIGHT_COLORS[0]}' opacity='0.9'/><text x='576' y='0'>h0</text>
          <circle cx='620' cy='-5' r='9' fill='{LIGHT_COLORS[1]}' opacity='0.9'/><text x='634' y='0'>h1</text>
        </g>
      </svg>
    </div>
    """


def formula_flow_html(features, weight, target_node, layer, pre_update, post_update):
    rows = message_table_for_target(features, weight, target_node)
    source_cards = []
    for row in rows:
        source_cards.append(f"""
        <div class='message-row'>
          <div class='source-badge'>{row['source']}</div>
          {vector_chip(row['hidden'], 'H source')}
          <div class='arrow'>× W</div>
          {vector_chip(row['after_w'], 'after W')}
          <div class='arrow'>× {row['edge_weight']:.2f}</div>
          {vector_chip(row['message'], 'message')}
        </div>
        """)
    summed = pre_update[target_node].detach().cpu().numpy()
    updated = post_update[target_node].detach().cpu().numpy()
    return f"""
    <div class='formula-board'>
      <div class='big-formula'>
        <span class='formula-token target-token'>H<sup>({layer + 1})</sup><sub>{target_node}</sub></span>
        <span class='equals'>=</span>
        <span class='formula-token relu-token'>ReLU</span>
        <span class='paren'>(</span>
        <span class='formula-token edge-token'>A_hat<sub>{target_node},j</sub></span>
        <span class='formula-token state-token'>H<sup>({layer})</sup><sub>j</sub></span>
        <span class='formula-token weight-token'>W<sup>({layer})</sup></span>
        <span class='paren'>)</span>
      </div>
      <div class='flow-title'>For target node {target_node}, each incoming neighbour creates one message:</div>
      {''.join(source_cards)}
      <div class='combine-row'>
        <div class='combine-box'>sum messages<br>{vector_chip(summed, 'pre-ReLU sum')}</div>
        <div class='big-arrow'>→</div>
        <div class='combine-box'>clip negative values<br>{vector_chip(updated, 'new target state')}</div>
      </div>
    </div>
    """


def render_message_passing_app():
    with app_output:
        clear_output(wait=True)
        layer = app_state["layer"]
        features = app_state["features"]
        target = target_dropdown.value
        weight = APP_WEIGHTS[min(layer, len(APP_WEIGHTS) - 1)]
        transformed = features @ weight
        pre_update = A_hat @ transformed
        post_update = torch.relu(pre_update)
        just_updated = app_state.get("last_update")

        if layer >= len(APP_WEIGHTS):
            update_button.description = "Start over"
            update_button.icon = "undo"
            status_text = "The demonstration layers are complete. Click Start over or choose a new target to reset to H^(0)."
        else:
            update_button.description = "Update one layer"
            update_button.icon = "refresh"
            status_text = "Previewing the next update. Red animated arrows show the exact messages used for the selected target."
        if just_updated is not None and layer < len(APP_WEIGHTS):
            status_text = f"Applied layer {just_updated['from_layer']} -> {just_updated['to_layer']} for target node {just_updated['target']}. The graph now shows H^({layer})."

        explanation = f"""
        <style>
          .gnn-shell {{ display: grid; grid-template-columns: 58% 42%; gap: 14px; width: 100%; align-items: start; }}
          .gnn-card {{ border: 1px solid #d1d5db; border-radius: 8px; background: #ffffff !important; color: #111827 !important; box-sizing: border-box; }}
          .gnn-card {{ padding: 12px; height: 590px; overflow-y: auto; }}
          .gnn-card * {{ color: #111827 !important; }}
          .status {{ background:#fff4cc !important; border-left:4px solid #b88700; padding:8px; margin-bottom:10px; font-weight: 800; }}
          .panel-title {{ font-size: 18px; font-weight: 900; margin-bottom: 10px; }}
          .big-formula {{ display:flex; align-items:center; flex-wrap:wrap; gap:6px; margin: 10px 0 12px 0; }}
          .formula-token {{ display:inline-block; border-radius:8px; padding:7px 9px; font-weight:900; border:2px solid #111827; background:#fff; }}
          .target-token {{ background:#fee2e2 !important; border-color:#d62728; }}
          .relu-token {{ background:#dcfce7 !important; border-color:#16a34a; }}
          .edge-token {{ background:#fef3c7 !important; border-color:#d4a000; }}
          .state-token {{ background:#e0f2fe !important; border-color:#0284c7; }}
          .weight-token {{ background:#ede9fe !important; border-color:#7c3aed; }}
          .equals, .paren {{ font-weight:900; font-size:20px; }}
          .flow-title {{ font-weight:800; margin: 8px 0; }}
          .message-row {{ display:grid; grid-template-columns: 1.15fr 1fr .42fr 1fr .55fr 1fr; gap:7px; align-items:center; border-bottom:1px solid #e5e7eb; padding:7px 0; }}
          .source-badge {{ font-weight:900; font-size:12px; border-left:5px solid #f0c419; padding-left:6px; }}
          .arrow, .big-arrow {{ font-weight:900; text-align:center; color:#991b1b !important; }}
          .vector-chip {{ border:1px solid #d1d5db; border-radius:7px; padding:5px; background:#f9fafb !important; text-align:center; }}
          .mini-title {{ font-size:10px; font-weight:900; color:#475569 !important; }}
          .mini-num {{ font-size:10px; font-weight:800; margin-top:2px; }}
          .vector-dot {{ display:inline-block; width:18px; height:18px; border-radius:50%; border:1px solid #111827; margin:2px; vertical-align:middle; }}
          .combine-row {{ display:grid; grid-template-columns: 1fr 44px 1fr; align-items:center; gap:8px; margin-top:12px; }}
          .combine-box {{ border:2px solid #111827; border-radius:8px; padding:8px; background:#ffffff !important; font-weight:900; text-align:center; }}
        </style>
        <div class='gnn-card'>
          <div class='panel-title'>Operation Explanation</div>
          <div class='status'>{status_text}</div>
          {formula_flow_html(features, weight, target, layer, pre_update, post_update)}
        </div>
        """

        left_panel = widgets.Output(layout=widgets.Layout(width="58%"))
        right_panel = widgets.Output(layout=widgets.Layout(width="42%"))
        display(widgets.HBox([left_panel, right_panel], layout=widgets.Layout(width="100%", align_items="flex-start")))

        with left_panel:
            display(HTML(single_graph_html(features, target, layer)))

        with right_panel:
            display(HTML(explanation))


def update_one_layer(_):
    layer = app_state["layer"]
    if layer >= len(APP_WEIGHTS):
        reset_app(None)
        return
    target = target_dropdown.value
    weight = APP_WEIGHTS[layer]
    app_state["features"] = torch.relu(A_hat @ (app_state["features"] @ weight))
    app_state["layer"] = layer + 1
    app_state["last_update"] = {"from_layer": layer, "to_layer": layer + 1, "target": target}
    render_message_passing_app()


def reset_app(_):
    app_state["layer"] = 0
    app_state["features"] = X0.clone()
    app_state["last_update"] = None
    render_message_passing_app()


def target_changed(_):
    app_state["layer"] = 0
    app_state["features"] = X0.clone()
    app_state["last_update"] = None
    render_message_passing_app()


update_button.on_click(update_one_layer)
reset_button.on_click(reset_app)
target_dropdown.observe(target_changed, names="value")

controls = widgets.HBox([target_dropdown, update_button, reset_button])
display(widgets.VBox([controls, app_output]))
render_message_passing_app()


## Step 1: Add Self-Loops and Normalize Edges

A GCN keeps a node's own information through a self-loop and normalizes messages by node degree:

\[
\hat{A}_{ij} = \frac{1}{\sqrt{d_i d_j}}
\]

This prevents high-degree nodes from dominating simply because they have many edges.

In [ ]:
display(pd.DataFrame(A_with_self_loops.numpy()).astype(int).style.set_caption("Adjacency with self-loops"))
display(pd.DataFrame(A_hat.numpy()).round(3).style.set_caption("Normalized adjacency A_hat"))

edge_weight_labels = {}
for i, j in G_toy.edges:
    edge_weight_labels[(i, j)] = f"{A_hat[i, j]:.2f}"

draw_graph_state(X0, "Graph with normalized edge weights", edge_weights=edge_weight_labels)

## Step 2: Apply Learnable Weights

Before aggregation, every node feature vector is transformed by the learnable weight matrix `W`. This is the neural-network part of the layer.

In [ ]:
display(pd.DataFrame(W0.numpy(), index=["input_feature_0", "input_feature_1"], columns=["hidden_0", "hidden_1"]).round(3).style.set_caption("Weight matrix W"))

XW = X0 @ W0
feature_table(XW, "Transformed node features X @ W")
draw_graph_state(XW, "After applying learnable weights: X @ W")

## Step 3: Aggregate Neighbour Messages

Now each node receives a weighted sum of transformed features from itself and its neighbours. This is where a GNN differs from a standard MLP.

In [ ]:
H_pre = A_hat @ XW
H1 = torch.relu(H_pre)

feature_table(H_pre, "Before ReLU: A_hat @ X @ W")
feature_table(H1, "After ReLU: H1")

draw_graph_state(H1, "Updated node states after one GCN-style layer")

In [ ]:
def explain_node_update(target_node):
    rows = []
    for source_node in range(X0.size(0)):
        coeff = A_hat[target_node, source_node].item()
        if coeff == 0:
            continue
        transformed = XW[source_node]
        contribution = coeff * transformed
        rows.append({
            "target": node_names[target_node],
            "source": node_names[source_node],
            "normalized_weight": coeff,
            "source_hidden_0": transformed[0].item(),
            "source_hidden_1": transformed[1].item(),
            "message_0": contribution[0].item(),
            "message_1": contribution[1].item(),
        })
    df = pd.DataFrame(rows)
    display(df.round(3))
    print("sum of messages:", H_pre[target_node].detach().numpy().round(3))
    print("after ReLU:", H1[target_node].detach().numpy().round(3))


explain_node_update(target_node=3)

## Step 4: Stack Another Layer

A second GNN layer lets information move two hops. The visualization below shows how representations become smoother across connected neighbourhoods.

In [ ]:
W1 = torch.tensor([
    [0.80, 0.35],
    [0.25, 0.90],
], dtype=torch.float32)

H2_pre = A_hat @ (H1 @ W1)
H2 = torch.relu(H2_pre)

feature_table(H2, "After two GCN-style layers")
draw_graph_state(H2, "Updated node states after two layers")

# Part B: Real Data Comparison - Feature-Only ML vs GNN

The goal is to make a clear teaching comparison:

- **Feature-only MLP:** sees each node's feature vector, but ignores graph edges.
- **GCN:** sees the same node features and additionally uses the graph.

On citation networks such as Cora, neighbouring papers often have related topics. A GNN can exploit this neighbourhood structure.

In [ ]:
dataset = Planetoid(root="/content/pyg_data", name="Cora")
data = dataset[0].to(DEVICE)

print(dataset)
print("nodes:", data.num_nodes)
print("edges:", data.num_edges)
print("node features:", dataset.num_node_features)
print("classes:", dataset.num_classes)
print("train/val/test:", int(data.train_mask.sum()), int(data.val_mask.sum()), int(data.test_mask.sum()))

In [ ]:
class FeatureOnlyMLP(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.lin1 = nn.Linear(in_channels, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index=None):
        x = self.lin1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin2(x)


class SimpleGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)

In [ ]:
@torch.no_grad()
def evaluate(model, data):
    model.eval()
    logits = model(data.x, data.edge_index)
    pred = logits.argmax(dim=-1)
    result = {}
    for split, mask in [
        ("train", data.train_mask),
        ("val", data.val_mask),
        ("test", data.test_mask),
    ]:
        acc = (pred[mask] == data.y[mask]).float().mean().item()
        result[f"{split}_acc"] = acc
    return result


def train_model(model, data, epochs=200, lr=0.01, weight_decay=5e-4):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(data.x, data.edge_index)
        loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        metrics = evaluate(model, data)
        metrics["epoch"] = epoch
        metrics["loss"] = loss.item()
        history.append(metrics)
    return model, pd.DataFrame(history)

In [ ]:
torch.manual_seed(SEED)
mlp = FeatureOnlyMLP(dataset.num_node_features, hidden_channels=64, out_channels=dataset.num_classes)
mlp, hist_mlp = train_model(mlp, data, epochs=200)

torch.manual_seed(SEED)
gcn = SimpleGCN(dataset.num_node_features, hidden_channels=64, out_channels=dataset.num_classes)
gcn, hist_gcn = train_model(gcn, data, epochs=200)

summary = pd.DataFrame([
    {"model": "Feature-only MLP", **evaluate(mlp, data)},
    {"model": "GCN with neighbourhood", **evaluate(gcn, data)},
])
display(summary.round(3))

In [ ]:
def plot_training_curves(hist_mlp, hist_gcn):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist_mlp["epoch"], hist_mlp["loss"], label="Feature-only MLP")
    axes[0].plot(hist_gcn["epoch"], hist_gcn["loss"], label="GCN")
    axes[0].set_title("Training loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()

    axes[1].plot(hist_mlp["epoch"], hist_mlp["test_acc"], label="Feature-only MLP")
    axes[1].plot(hist_gcn["epoch"], hist_gcn["test_acc"], label="GCN")
    axes[1].set_title("Test accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    plt.show()


plot_training_curves(hist_mlp, hist_gcn)

## Inspect One Prediction and Its Neighbourhood

The next cell selects a test node where the GCN prediction is correct and the feature-only MLP is wrong, if such a node exists. Then it draws the node's 2-hop neighbourhood.

In [ ]:
@torch.no_grad()
def predictions(model):
    model.eval()
    return model(data.x, data.edge_index).argmax(dim=-1).cpu()

mlp_pred = predictions(mlp)
gcn_pred = predictions(gcn)
y_true = data.y.cpu()
test_nodes = data.test_mask.cpu().nonzero(as_tuple=False).view(-1)

candidates = [
    int(node) for node in test_nodes
    if mlp_pred[node] != y_true[node] and gcn_pred[node] == y_true[node]
]

if candidates:
    focus_node = candidates[0]
    print("Selected a test node where GCN is correct and MLP is wrong:", focus_node)
else:
    focus_node = int(test_nodes[0])
    print("No MLP-wrong/GCN-correct node found in this run; showing first test node:", focus_node)

print("true class:", int(y_true[focus_node]))
print("MLP prediction:", int(mlp_pred[focus_node]))
print("GCN prediction:", int(gcn_pred[focus_node]))

In [ ]:
def draw_k_hop_neighbourhood(node_id, num_hops=2):
    subset, edge_index_sub, mapping, edge_mask = k_hop_subgraph(
        node_id,
        num_hops=num_hops,
        edge_index=data.edge_index.cpu(),
        relabel_nodes=True,
    )
    edges = [tuple(edge) for edge in edge_index_sub.t().tolist()]
    G = nx.Graph()
    G.add_edges_from(edges)
    if not G.nodes:
        G.add_node(0)
    pos = nx.spring_layout(G, seed=SEED)

    colors = []
    for local_node in G.nodes:
        original = int(subset[local_node]) if local_node < len(subset) else node_id
        if original == node_id:
            colors.append("#d62728")
        elif int(y_true[original]) == int(y_true[node_id]):
            colors.append("#2ca02c")
        else:
            colors.append("#cccccc")

    labels = {local: str(int(subset[local])) for local in G.nodes if local < len(subset)}
    plt.figure(figsize=(8, 6))
    nx.draw_networkx_edges(G, pos, alpha=0.35)
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=260, edgecolors="#222")
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=8)
    plt.title(f"{num_hops}-hop neighbourhood around node {node_id}\\nred = focus node, green = same class")
    plt.axis("off")
    plt.show()


draw_k_hop_neighbourhood(focus_node, num_hops=2)

## Teaching Takeaway

A standard MLP treats every node independently after reading its feature vector. It can learn from features of a paper, but it cannot directly use who cites whom.

A GNN uses the same node features **and** the graph. In citation data, connected papers often share topics. In spatial data, neighbouring cells, nearby street images, adjacent road segments, or spatially connected observations often carry related information. That is why the GNN comparison is a useful bridge to geospatial AI.